# Importa Bibliotecas

In [1]:
import glob
import ast

import pandas as pd
import numpy  as np

from unidecode  import unidecode

# Trata dados Brutos VivaReal

In [2]:
path = r'D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\*.parquet'

# Read all parquet files and concatenate them into a single DataFrame
all_files = glob.glob(path)
dataframes = []  # List to store successfully read DataFrames

for file in all_files:
    try:
        df = pd.read_parquet(file)
        dataframes.append(df)
        print(f"Successfully read {file}")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all successfully read DataFrames
if dataframes:
    vivareal = pd.concat(dataframes, ignore_index=True)
else:
    print("No files were read successfully.")

Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abadia de Goias.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abadia dos Dourados.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abadiania.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abaete.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abaetetuba.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abaira.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abare.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abatia.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Scraping Vivareal\Abrantes.parquet
Successfully read D:\Drive\Codigos bee6\bee6_arquivo

In [7]:
vivareal['description']

0          Trata-se de uma Ã¡rea comercial com mais ou me...
1          A tÃ©rrea no bairro Setor Jardim Nova Abadia c...
2          Casa em Abadia de GoiÃ¡s <br><br>Terreno 183m<...
3          Lotes Prontos P/ Construir 276mÂ²<br>Parcelas ...
4          NOME DO EMPREENDIMENTO: RESIDENCIAL ORTEGAL<br...
                                 ...                        
5975925    Oportunidade! Casa com 59,04mÂ² de Ãrea Priva...
5975926    ImÃ³vel de propriedade do Banco disponibilizad...
5975927    Oportunidade! Casa com 42,70 mÂ² Privativos e ...
5975928    ImÃ³vel de propriedade do Banco disponibilizad...
5975929    .Baixar matrÃ­cula do imÃ³velBaixar edital e a...
Name: description, Length: 5975930, dtype: object

# Gera Vivareal para precificação

In [8]:
# Relacao de colunas importantes a serem extraidas
columns_dict = {'unit_types': 'Tipo','unit_sub_types': 'SubTipo','usage_types': 'Tipo_Uso',
    'parking_spaces': 'Num_Vagas_Est','zipcode': 'Cep','ibge_city_id': 'Municipio',
    'bathrooms': 'Num_Banheiros','total_areas': 'Area_Total_m2','bedrooms': 'Num_Quartos',
    'suites': 'Num_Suites','usable_areas' : 'Area_Construida_m2','id':'idImovel',
    'pricing_infos':'pricing_infos','location_id':'location_id',
    'point_lon' : 'Long','point_lat' : 'Lat'}

keys_list = list(columns_dict.keys())

# Extrai as colunas e renomeia
vivareal = vivareal[keys_list]
vivareal.rename(columns=columns_dict, inplace=True)

# Dropa duplicados
vivareal.drop_duplicates(subset = 'idImovel', inplace = True)

# Trata a Municipio
vivareal['Municipio'] = vivareal['location_id'].apply(lambda x: x.split('>NULL>')[1].split('>')[0])

# Extrai todas as colunas da pricing.info
def convert_pricing_info(pricing_info):
    try:
        pricing_info_list = ast.literal_eval(pricing_info)
        if isinstance(pricing_info_list, list) and len(pricing_info_list) > 0:
            return pricing_info_list[0]
    except (SyntaxError, ValueError):
        pass  # Handle invalid JSON or other exceptions here
    return None

# Apply the conversion function
vivareal['pricing_infos_dict'] = vivareal['pricing_infos'].apply(convert_pricing_info)

# Extract 'price', 'yearlyIptu', 'businessType', and 'monthlyCondoFee' into separate columns
vivareal['Preco'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('price') if x else None)
vivareal['Iptu_Anual'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('yearlyIptu') if x else None)
vivareal['Tipo_Negocio'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('businessType') if x else None)
vivareal['Condominio_Mensal'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('monthlyCondoFee') if x else None)

# Drop the intermediate column and rows with missing 'Preco' values
vivareal.drop(['pricing_infos', 'pricing_infos_dict'], axis=1, inplace=True)
vivareal.dropna(subset=['Preco'], inplace=True)

# Remove os colchetes das colunas
columns_to_clean = ['Tipo', 'SubTipo', 'Tipo_Uso', 'Num_Vagas_Est', 'Num_Banheiros',
                     'Area_Total_m2', 'Num_Quartos', 'Num_Suites','Area_Construida_m2']
for column in columns_to_clean:
    vivareal[column] = vivareal[column].astype(str).str.replace('[','').str.replace(']','')

# Converte tudo pra float
columns_to_convert = ['Num_Vagas_Est', 'Num_Banheiros', 'Area_Total_m2', 'Num_Quartos', 
                      'Num_Suites', 'Area_Construida_m2','Preco']

for column in columns_to_convert:
    vivareal[column] = vivareal[column].replace('', np.nan)
    vivareal[column] = vivareal[column].str.replace("'", "").str.replace(',', '.').astype(float, errors='ignore')

#Arruma o CEP q veio como float
vivareal['Cep'] = vivareal['Cep'].apply(lambda x: '{:08d}'.format(int(x)) if pd.notnull(x) else np.nan)
vivareal['Cep'] = vivareal['Cep'].apply(lambda x: x[:5] + '-' + x[5:] if pd.notnull(x) else np.nan)

# Junta as lat long numa mesma coluna

vivareal['Lat_Long'] = vivareal.apply(lambda row: f"{row['Lat']},{row['Long']}", axis=1)
vivareal['Lat_Long'] = vivareal['Lat_Long'].replace('nan,nan', np.nan)

In [10]:
vivareal.dtypes

Tipo                   object
SubTipo                object
Tipo_Uso               object
Num_Vagas_Est          object
Cep                    object
Municipio              object
Num_Banheiros          object
Area_Total_m2         float64
Num_Quartos            object
Num_Suites             object
Area_Construida_m2     object
idImovel                int64
location_id            object
Long                  float64
Lat                   float64
Preco                 float64
Iptu_Anual             object
Tipo_Negocio           object
Condominio_Mensal      object
Lat_Long               object
dtype: object

In [ ]:


# Reorganiza colunas
vivareal = vivareal[['Tipo','SubTipo','Tipo_Uso','Municipio','Cep',
                     'Area_Construida_m2','Area_Total_m2','Condominio_Mensal',
                     'Iptu_Anual','Num_Quartos',
                     'Num_Suites','Num_Banheiros','Num_Vagas_Est',
                     'Preco','Tipo_Negocio','idImovel','Lat_Long']]

# Subdivide os dataframes

# DataFrames for Sao Paulo
vivareal_aluguel_sp = vivareal[(vivareal['Tipo_Negocio'] == 'RENTAL') & (vivareal['Municipio'] == 'Sao Paulo')].to_parquet('D:\Drive\Arquivos Codigo\\bee6\VivaReal\Tratado\VivaReal_Aluguel_Sp.parquet')
vivareal_venda_sp = vivareal[(vivareal['Tipo_Negocio'] == 'SALE') & (vivareal['Municipio'] == 'Sao Paulo')].to_parquet('D:\Drive\Arquivos Codigo\\bee6\VivaReal\Tratado\VivaReal_Venda_Sp.parquet')

# DataFrames for Brazil excluding Sao Paulo
vivareal_aluguel_br = vivareal[(vivareal['Tipo_Negocio'] == 'RENTAL') & (vivareal['Municipio'] != 'Sao Paulo')].to_parquet('D:\Drive\Arquivos Codigo\\bee6\VivaReal\Tratado\VivaReal_Aluguel_Br.parquet')
vivareal_venda_br = vivareal[(vivareal['Tipo_Negocio'] == 'SALE') & (vivareal['Municipio'] != 'Sao Paulo')].to_parquet('D:\Drive\Arquivos Codigo\\bee6\VivaReal\Tratado\VivaReal_Venda_Br.parquet')

C:\Users\Max Power\AppData\Local\Temp\ipykernel_6916\1902093240.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vivareal.rename(columns=columns_dict, inplace=True)
C:\Users\Max Power\AppData\Local\Temp\ipykernel_6916\1902093240.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vivareal.drop_duplicates(subset = 'idImovel', inplace = True)
C:\Users\Max Power\AppData\Local\Temp\ipykernel_6916\1902093240.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydat

# Gera Vivareal para busca (Urban Space)

In [4]:
# Cria a municipio
vivareal['Municipio'] = vivareal['location_id'].apply(lambda x: x.split('>NULL>')[1].split('>')[0])

# Seleciona colunas relevantes para a bee
vivareal = vivareal[['amenities','usable_areas','listing_type','description','title','unit_types',
          'unit_sub_types','id','parking_spaces','zipcode','street_number','point_lon',
          'point_lat','street','location_id','neighborhood','suites','bathrooms','usage_types',
          'total_areas','bedrooms','pricing_infos','advertiser_contact_phones','whatsapp_number',
          'account_name','account_license_number','account_phones_primary','account_phones_mobile',
          'Municipio']]

# Extrai todas as colunas da pricing.info
def convert_pricing_info(pricing_info):
    try:
        pricing_info_list = ast.literal_eval(pricing_info)
        if isinstance(pricing_info_list, list) and len(pricing_info_list) > 0:
            return pricing_info_list[0]
    except (SyntaxError, ValueError):
        pass  # Handle invalid JSON or other exceptions here
    return None

# Apply the conversion function
vivareal['pricing_infos_dict'] = vivareal['pricing_infos'].apply(convert_pricing_info)

# Extract 'price', 'yearlyIptu', 'businessType', and 'monthlyCondoFee' into separate columns
vivareal['Preco'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('price') if x else None)
vivareal['Iptu_Anual'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('yearlyIptu') if x else None)
vivareal['Tipo_Negocio'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('businessType') if x else None)
vivareal['Condominio_Mensal'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('monthlyCondoFee') if x else None)

vivareal.drop(['pricing_infos', 'pricing_infos_dict'], axis=1, inplace=True)

# Remove os colchetes das colunas
columns_to_clean = ['unit_types', 'unit_sub_types', 'usage_types', 'parking_spaces', 'bathrooms',
                     'total_areas', 'bedrooms', 'suites','usable_areas']
for column in columns_to_clean:
    vivareal[column] = vivareal[column].astype(str).str.replace('[','').str.replace(']','')

#Arruma o CEP q veio como float
vivareal['zipcode'] = vivareal['zipcode'].apply(lambda x: '{:08d}'.format(int(x)) if pd.notnull(x) else np.nan)
vivareal['zipcode'] = vivareal['zipcode'].apply(lambda x: x[:5] + '-' + x[5:] if pd.notnull(x) else np.nan)

# Cria uma coluna de endereco unica
vivareal['Endereco'] = vivareal['street'].astype(str) + ', ' + vivareal['street_number'].astype(str)
vivareal['Endereco'] = vivareal['Endereco'].str.replace('.0','')

# Cria lat long 
vivareal['Lat_Long'] = vivareal['point_lat'].astype(float).astype(str) + ', ' + vivareal['point_lon'].astype(float).astype(str)

# Renomeia todas as colunas relevantes
columns_dict = {'unit_types': 'Tipo',
                'unit_sub_types': 'SubTipo',
                'usage_types': 'Tipo_Uso',
                'parking_spaces': 'Num_Vagas_Est',
                'zipcode': 'Cep',
                'bathrooms': 'Num_Banheiros',
                'total_areas': 'Area_Total_m2',
                'bedrooms': 'Num_Quartos',
                'suites': 'Num_Suites',
                'neighborhood':'Bairro',
                'usable_areas' : 'Area_Construida_m2',
                'id':'idImovel','amenities': 'Amenidades',
                'listing_type': 'Condicao',
                'description': 'Descricao',
                'title': 'Titulo',
                'account_name': 'Proprietario',
                'account_license_number':'Creci',
                'neighborhood': 'Bairro',
                'advertiser_contact_phones':'Telefones'}

existing_columns = [col for col in vivareal.columns if col not in columns_dict.keys()]
vivareal.rename(columns=columns_dict, inplace=True)
vivareal = vivareal[list(columns_dict.values()) + existing_columns]

# Gera a coluna WhatsApp
base_url = 'https://api.whatsapp.com/send?phone=55'
phone_number = vivareal['whatsapp_number'].astype(str).str.split('.').str[0]
message = 'Olá, ' + vivareal['Proprietario'] + ', tudo bem?'
encoded_message = message.str.replace(' ', '%20')
vivareal['WhatsApp'] = base_url + phone_number + '&text=' + encoded_message

# Dropa as irrelevantes
columns_to_drop = ['street_number','street','location_id',
                   'whatsapp_number','account_phones_primary',
                   'account_phones_mobile']

vivareal.drop(columns=columns_to_drop, inplace=True)

# Trata todas as colunas float
#vivareal['Num_Vagas_Est'] = vivareal['Num_Vagas_Est'].str.replace('[^\d.]', '', regex=True).astype(float)
#vivareal['Num_Banheiros'] = vivareal['Num_Banheiros'].str.replace('[^\d.]', '', regex=True).astype(float)
#vivareal['Num_Quartos'] = vivareal['Num_Quartos'].str.replace('[^\d.]', '', regex=True).astype(float)
#vivareal['Num_Suites'] = vivareal['Num_Suites'].str.replace('[^\d.]', '', regex=True).astype(float)
#vivareal['Area_Construida_m2'] = vivareal['Area_Construida_m2'].str.replace('[^\d.]', '', regex=True).astype(float)

C:\Users\Max Power\AppData\Local\Temp\ipykernel_18452\1647974264.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vivareal['pricing_infos_dict'] = vivareal['pricing_infos'].apply(convert_pricing_info)
C:\Users\Max Power\AppData\Local\Temp\ipykernel_18452\1647974264.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vivareal['Preco'] = vivareal['pricing_infos_dict'].apply(lambda x: x.get('price') if x else None)
C:\Users\Max Power\AppData\Local\Temp\ipykernel_18452\1647974264.py:27: SettingWithCopyWarn